In [ ]:
"""
Objective: Use the Autoregressive Distributed Lag (ARDL) method and Pesaran's Bounds Test 
to estimate Error Correction Models (ECM) and map the short- and long-term macroeconomic 
determinants for each credit segment.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.ardl import ardl_select_order, UECM
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING AND SAMPLE TRUNCATION
# ==========================================
def load_ardl_data(csv_file: str) -> pd.DataFrame:
    """Loads the data and filters from 2005 onwards for level-test stability."""
    print("[*] Loading clean dataset...")
    df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
    df_model = df.loc['2005-01-01':'2026-04-01'].copy()
    return df_model

# ==========================================
# 2. MODEL SPECIFICATION AND ECM ESTIMATION
# ==========================================
def estimate_ardl_ecm(df: pd.DataFrame, specifications: dict):
    """
    Performs automatic lag selection (BIC), estimates the Unrestricted Error Correction Model (UECM), 
    and extracts long-run (Bounds Test) and short-run coefficients.
    """
    print("\n[*] Starting automatic ARDL selection and ECM estimation...\n")
    
    for portfolio_name, specs in specifications.items():
        print(f"=======================================================")
        print(f" PORTFOLIO: {portfolio_name.upper()}")
        print(f"=======================================================")
        
        y = df[specs['y']].dropna()
        X = df[specs['x']].dropna()
        
        # Index synchronization
        common_index = y.index.intersection(X.index)
        y, X = y.loc[common_index], X.loc[common_index]
        
        # Strict lag order selection (BIC)
        selection = ardl_select_order(endog=y, maxlag=4, exog=X, maxorder=2, ic='bic', trend='c')
        
        if not selection.dl_lags:
            print("[!] No macro variable survived the BIC filter.")
            continue
        
        order_x_dict = {col: max(1, max(lags)) for col, lags in selection.dl_lags.items()}
        X_selected = X[list(selection.dl_lags.keys())]
        order_y = max(1, max(selection.ar_lags)) if selection.ar_lags is not None else 1
        
        print(f"[*] Best architecture (BIC): ARDL({order_y}, {order_x_dict})")
        
        # UECM Instantiation
        ecm_model = UECM(endog=y, lags=order_y, exog=X_selected, order=order_x_dict, trend='c')
        ecm_fit = ecm_model.fit()
        
        # Pesaran's Bounds Test
        bounds_test = ecm_fit.bounds_test(case=3)
        
        # Error Correction Term (ECT) extraction
        ect_key = specs['y'] + '.L1'
        if ect_key not in ecm_fit.params:
            ect_key = [c for c in ecm_fit.params.index if '.L1' in c and specs['y'] in c][0]
            
        ect_param = ecm_fit.params[ect_key]
        ect_pvalue = ecm_fit.pvalues[ect_key]
        
        print(f"[*] Error Correction Term (Lambda/ECT): {ect_param:.4f} (P-value: {ect_pvalue:.4f})")
        print(f"[*] Bounds Test F-statistic: {bounds_test.stat:.2f}")
        
        if ect_param < 0 and ect_pvalue < 0.05:
            ecm_half_life = np.log(0.5) / ect_param
            print(f"[*] Status: Cointegration validated. Shocks dissipate by half in {ecm_half_life:.2f} quarters.")
        else:
            print("[!] Warning: Unstable ECT or lack of robust cointegration at the 5% level.")
            
        print("\n--- Main Short-Term Determinants (First Differences) ---")
        short_run = ecm_fit.params.filter(like='D.')
        pvalues_sr = ecm_fit.pvalues.filter(like='D.')
        
        for var, coef in short_run.items():
            if pvalues_sr[var] < 0.10: 
                print(f" > {var}: {coef:.4f} (Sig: {pvalues_sr[var]:.3f})")
        print("\n")

if __name__ == "__main__":
    df_model = load_ardl_data('clean_macro_fred_dataset.csv')
    
    model_specifications = {
        'Mortgages': {
            'y': 'delinq_mortgages',
            'x': ['unemployment', 'fed_funds', 'spread_10y3m', 'home_prices', 'debt_service_ratio', 'sloos_tightening']
        },
        'Consumer': {
            'y': 'delinq_consumer',
            'x': ['unemployment', 'fed_funds', 'spread_10y3m', 'vehicle_sales', 'debt_service_ratio', 'sloos_tightening']
        },
        'Credit Cards': {
            'y': 'delinq_credit_cards',
            'x': ['unemployment', 'fed_funds', 'spread_10y3m', 'debt_service_ratio', 'sloos_tightening']
        }
    }
    
    estimate_ardl_ecm(df_model, model_specifications)
    print("[SUCCESS] ARDL-ECM pipeline completed.")